# Solar Flux Prediction

In [1]:
import torch

device_try = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Selected device: {device_try}")

Selected device: cuda


In [2]:
import os

print("=== TEST TMUX ===")
tmux_var = os.environ.get("TMUX")

if tmux_var:
    print("STATUS: Jupyter server is running in a TMUX session!")
    print(f"Tmux socket path: {tmux_var}")
else:
    print("STATUS: The Jupyter server is NOT inside Tmux (it runs in the global terminal).")

=== TEST TMUX ===
STATUS: Jupyter server is running in a TMUX session!
Tmux socket path: /tmp//tmux-1022/default,842946,5


In [3]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [4]:
print("--- Caricamento e Aggregazione Giornaliera ---")
df = pd.read_csv("../Data/MARSIS_historical_dataset.csv", sep=";")
df.columns = df.columns.str.strip()

# 1. Conversione tempo
df['datetime'] = pd.to_datetime(df['FM_data_ephemeris_time'], unit='s', origin=pd.Timestamp('2000-01-01 12:00:00'))

# 2. Aggregazione Giornaliera
df_daily = df.groupby(df['datetime'].dt.to_period('D')).agg({
    'FM_data_F10_7_index': 'mean'
}).reset_index()

df_daily['datetime'] = df_daily['datetime'].dt.to_timestamp()

# 3. FIX: Riempimento giorni mancanti PRIMA di estrarre le feature temporali
df_daily = df_daily.set_index('datetime').asfreq('D', method='ffill').reset_index()

# 4. Estrazione feature temporali (ora su un calendario perfetto e senza buchi)
df_daily['day_of_year'] = df_daily['datetime'].dt.dayofyear
df_daily['month_of_year'] = df_daily['datetime'].dt.month 

print(f"Dataset giornaliero creato. Totale giorni disponibili: {len(df_daily)}")

--- Caricamento e Aggregazione Giornaliera ---
Dataset giornaliero creato. Totale giorni disponibili: 5532


In [5]:
import numpy as np
from time_series_utils import *

LOOKBACK_WINDOW = 30 

# Creiamo le matrici usando il dataset giornaliero e il giorno dell'anno
X_flux, X_time, y = create_dataset_windows(
    data=df_daily, 
    target_col='FM_data_F10_7_index', 
    time_col='day_of_year',  
    lookback=LOOKBACK_WINDOW
)

# Split Cronologico (80% Train, 20% Test)
split_idx = int(len(y) * 0.8)

X_flux_train, X_flux_test = X_flux[:split_idx], X_flux[split_idx:]
X_time_train, X_time_test = X_time[:split_idx], X_time[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

## Linear regression for solar flux prediction 

In [6]:
print("\n--- Training: Regressione Lineare ---")

# Uniamo le feature (Flusso_t-30...Flusso_t + Giorno_t-30...Giorno_t)
X_linear_train = np.hstack((X_flux_train, X_time_train))
X_linear_test = np.hstack((X_flux_test, X_time_test))

# Inizializzazione e fit
lr_model = LinearRegression()
lr_model.fit(X_linear_train, y_train)

# Predizione e Valutazione
y_pred_lr = lr_model.predict(X_linear_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
print(f"[Linear Regression] Test MAE: {mae_lr:.3f}, Test RMSE: {rmse_lr:.3f}")


--- Training: Regressione Lineare ---
[Linear Regression] Test MAE: 0.401, Test RMSE: 1.352


## LSTM for solar flux prediction

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

print("\n--- Training: PyTorch LSTM (Dati Giornalieri) ---")

# =====================================================================
# 1. NORMALIZZAZIONE DEI DATI
# =====================================================================
# Inizializziamo gli scaler dedicati
scaler_X_flux = MinMaxScaler()
scaler_X_time = MinMaxScaler()
scaler_y = MinMaxScaler()

# Fit sul train e transform su train e test
X_flux_train_scaled = scaler_X_flux.fit_transform(X_flux_train)
X_flux_test_scaled = scaler_X_flux.transform(X_flux_test)

X_time_train_scaled = scaler_X_time.fit_transform(X_time_train)
X_time_test_scaled = scaler_X_time.transform(X_time_test)

y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1))

# COSTRUIAMO L'INPUT 3D RICHIESTO DALLA LSTM: (Campioni, Finestra, Feature)
# Uniamo il flusso e il giorno dell'anno lungo l'ultimo asse (axis=-1)
X_lstm_train = np.stack((X_flux_train_scaled, X_time_train_scaled), axis=-1)
X_lstm_test = np.stack((X_flux_test_scaled, X_time_test_scaled), axis=-1)



--- Training: PyTorch LSTM (Dati Giornalieri) ---


### 100 epochs

In [11]:
from utils.train_pytorch_model import *
from models.LSTM import *

# Avviamo l'addestramento usando la tua funzione modificata
# Passiamo hidden_dim=32 alla fine; verrà catturato da **model_kwargs
trained_lstm, device, training_history = train_pytorch_model(
    X_train=X_lstm_train,         # I tuoi dati 3D (Campioni, 30, 2)
    y_train=y_train_scaled,       # Il target normalizzato
    input_dim=2,                  # Flusso solare + Giorno dell'anno
    model_class=LSTMRegressor,    # La classe del modello
    epochs=100,                   # Massimo numero di epoche
    patience=8,                   # Interrompi se la val_loss non migliora per 8 epoche
    batch_size=32,                # Dimensione del batch ottimale per dati giornalieri
    hidden_dim=32                 # <--- Parametro extra inoltrato alla LSTM!
)

# =====================================================================
# INFERENZA SUI DATI DI TEST (Utilizzando il modello addestrato)
# =====================================================================
trained_lstm.eval()
X_test_t = torch.tensor(X_lstm_test, dtype=torch.float32).to(device)

with torch.no_grad():
    y_pred_lstm_scaled = trained_lstm(X_test_t).cpu().numpy()

# De-normalizzazione per visualizzare i risultati in scala reale
y_pred_lstm = scaler_y.inverse_transform(y_pred_lstm_scaled).flatten()

# Calcolo delle metriche di performance finali
from sklearn.metrics import mean_absolute_error, mean_squared_error
mae_lstm = mean_absolute_error(y_test, y_pred_lstm)
rmse_lstm = np.sqrt(mean_squared_error(y_test, y_pred_lstm))

print(f"\n[LSTM Network] Test MAE: {mae_lstm:.3f}, Test RMSE: {rmse_lstm:.3f}")

Early stopping at epoch 60

[LSTM Network] Test MAE: 0.442, Test RMSE: 1.294


### 1000 epochs

In [7]:
from utils.train_pytorch_model import *
from models.LSTM import *

# Avviamo l'addestramento usando la tua funzione modificata
# Passiamo hidden_dim=32 alla fine; verrà catturato da **model_kwargs
trained_lstm_2, device_2, training_history_2 = train_pytorch_model(
    X_train=X_lstm_train,         # I tuoi dati 3D (Campioni, 30, 2)
    y_train=y_train_scaled,       # Il target normalizzato
    input_dim=2,                  # Flusso solare + Giorno dell'anno
    model_class=LSTMRegressor,    # La classe del modello
    epochs=1000,                   # Massimo numero di epoche
    patience=8,                   # Interrompi se la val_loss non migliora per 8 epoche
    batch_size=32,                # Dimensione del batch ottimale per dati giornalieri
    hidden_dim=128                 # <--- Parametro extra inoltrato alla LSTM!
)

# =====================================================================
# INFERENZA SUI DATI DI TEST (Utilizzando il modello addestrato)
# =====================================================================
trained_lstm_2.eval()
X_test_t_2 = torch.tensor(X_lstm_test, dtype=torch.float32).to(device_2)

with torch.no_grad():
    y_pred_lstm_scaled_2 = trained_lstm_2(X_test_t_2).cpu().numpy()

# De-normalizzazione per visualizzare i risultati in scala reale
y_pred_lstm_2 = scaler_y.inverse_transform(y_pred_lstm_scaled_2).flatten()

# Calcolo delle metriche di performance finali
from sklearn.metrics import mean_absolute_error, mean_squared_error
mae_lstm_2 = mean_absolute_error(y_test, y_pred_lstm_2)
rmse_lstm_2 = np.sqrt(mean_squared_error(y_test, y_pred_lstm_2))

print(f"\n[LSTM Network] Test MAE: {mae_lstm_2:.3f}, Test RMSE: {rmse_lstm_2:.3f}")

Early stopping at epoch 55

[LSTM Network] Test MAE: 0.403, Test RMSE: 1.288


## Saving models

In [ ]:
# Impacchettiamo gli scaler in un dizionario
my_scalers = {
    'scaler_X_flux': scaler_X_flux,
    'scaler_X_time': scaler_X_time,
    'scaler_y': scaler_y
}

save_lstm_and_scalers(
    model=trained_lstm, 
    scalers_dict=my_scalers, 
    save_dir="saved_lstm", 
    model_name="lstm_100_epochs.pth", 
    scaler_name="lstm_scalers_100_epochs.pkl"
)

# Chiamiamo la funzione
save_lstm_and_scalers(
    model=trained_lstm_2, 
    scalers_dict=my_scalers, 
    save_dir="saved_lstm", 
    model_name="lstm_1000_epochs.pth", 
    scaler_name="lstm_scalers_1000_epochs.pkl"
)